# EAGF Notebook 1: Final Results Demo

**Ethical AI Governance Framework (EAGF)** — Final Verified Results

This notebook demonstrates the EAGF framework's final verified results compared to the external AIF360-DP baseline:

1. Load pre-computed results from 10-seed runs (seeds 42-51)
2. Compute mean±std and 95% confidence intervals across paired seeds
3. Display final metrics: Accuracy, Recall Parity, Clarity, Privacy (corrected), Accountability, Trust Index
4. Show baseline vs EAGF comparison with TI_certified support
5. Verify +24.35% Trust Index improvement with statistical significance (p=0.005)

**Paper:** *Ethical AI Governance for Cybersecurity in RE-IoT Systems* (Jan et al., 2025)

---
**Runtime:** ~30 seconds (loads pre-computed results, no training)

[![GitHub](https://img.shields.io/badge/GitHub-aliakarma%2Feagf-blue?logo=github)](https://github.com/aliakarma/eagf)  **Repository:** [https://github.com/aliakarma/eagf](https://github.com/aliakarma/eagf)

In [ ]:
!git clone https://github.com/aliakarma/eagf.git
!cd eagf
!pip install -r eagf/requirements.txt

In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = str(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Using PROJECT_ROOT={PROJECT_ROOT}")

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(PROJECT_ROOT) if 'PROJECT_ROOT' in globals() else Path.cwd()
if not (PROJECT_ROOT / 'configs').exists() and (PROJECT_ROOT.parent / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = str(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import yaml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Environment ready.')
print(f'numpy {np.__version__}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')

## 1. Load Pre-Computed Results (10-Seed Runs)

In [ ]:
import json
from pathlib import Path

# Setup PROJECT_ROOT
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = str(Path.cwd())
if Path(PROJECT_ROOT).name != 'eagf':
    PROJECT_ROOT = str(Path(PROJECT_ROOT) / 'eagf')
import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Define seeds
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
BASELINE_DIR = Path(PROJECT_ROOT) / 'results' / 'baseline_aif360_dp'
EAGF_DIR = Path(PROJECT_ROOT) / 'results' / 'runs' / 'full'

print('Loading Pre-Computed Results')
print('=' * 50)
print(f'Baseline dir: {BASELINE_DIR}')
print(f'EAGF dir:     {EAGF_DIR}')
print(f'Seeds:        {SEEDS}')

# Find paired seeds (seeds that exist in both baseline and EAGF)
baseline_seeds = set()
eagf_seeds = set()

for seed_dir in BASELINE_DIR.glob('seed_*'):
    try:
        seed = int(seed_dir.name.replace('seed_', ''))
        results_file = seed_dir / 'results.json'
        if results_file.exists():
            baseline_seeds.add(seed)
    except:
        pass

for seed_dir in EAGF_DIR.glob('seed_*'):
    try:
        seed = int(seed_dir.name.replace('seed_', ''))
        results_file = seed_dir / 'results.json'
        if results_file.exists():
            eagf_seeds.add(seed)
    except:
        pass

paired_seeds = sorted(list(baseline_seeds & eagf_seeds & set(SEEDS)))
print(f'\nPaired seeds found: {paired_seeds}')
print(f'Total paired runs: {len(paired_seeds)}')


## 2. Aggregate Results Across 10 Paired Seeds

In [ ]:
import numpy as np
import json

# Load results for both baseline and EAGF across all paired seeds
baseline_results = {}
eagf_results = {}

for seed in paired_seeds:
    # Load baseline results
    baseline_file = BASELINE_DIR / f'seed_{seed}' / 'results.json'
    if baseline_file.exists():
        with open(baseline_file) as f:
            baseline_results[seed] = json.load(f)
    
    # Load EAGF results
    eagf_file = EAGF_DIR / f'seed_{seed}' / 'results.json'
    if eagf_file.exists():
        with open(eagf_file) as f:
            eagf_results[seed] = json.load(f)

print(f'\nLoaded {len(baseline_results)} baseline runs')
print(f'Loaded {len(eagf_results)} EAGF runs')

# Compute mean and std for key metrics
metrics = ['accuracy', 'recall_parity', 'clarity', 'privacy', 'accountability', 'trust_index', 'trust_index_certified']

baseline_stats = {}
eagf_stats = {}

for metric in metrics:
    baseline_vals = [baseline_results[s][metric] for s in paired_seeds if metric in baseline_results[s]]
    eagf_vals = [eagf_results[s][metric] for s in paired_seeds if metric in eagf_results[s]]
    
    baseline_stats[metric] = {
        'mean': np.mean(baseline_vals) if baseline_vals else 0.0,
        'std': np.std(baseline_vals) if baseline_vals else 0.0,
        'values': baseline_vals
    }
    
    eagf_stats[metric] = {
        'mean': np.mean(eagf_vals) if eagf_vals else 0.0,
        'std': np.std(eagf_vals) if eagf_vals else 0.0,
        'values': eagf_vals
    }

print('\nAggregation complete. Ready for display.')


## 3. Final Results Comparison (Baseline vs EAGF)

In [ ]:
import pandas as pd

# Build comparison dataframe
comparison_data = []

metric_labels = {
    'accuracy': 'Accuracy',
    'recall_parity': 'Recall Parity (RP)',
    'clarity': 'Clarity (C)',
    'privacy': 'Privacy (P)',
    'accountability': 'Accountability (A)',
    'trust_index': 'Trust Index (TI)',
    'trust_index_certified': 'TI_certified'
}

for metric in metrics:
    label = metric_labels.get(metric, metric)
    baseline_mean = baseline_stats[metric]['mean']
    baseline_std = baseline_stats[metric]['std']
    eagf_mean = eagf_stats[metric]['mean']
    eagf_std = eagf_stats[metric]['std']
    
    improvement = eagf_mean - baseline_mean
    improvement_pct = (improvement / baseline_mean * 100) if baseline_mean != 0 else 0
    
    comparison_data.append({
        'Metric': label,
        'Baseline (mean±std)': f'{baseline_mean:.4f}±{baseline_std:.4f}',
        'EAGF (mean±std)': f'{eagf_mean:.4f}±{eagf_std:.4f}',
        'Improvement': f'{improvement:+.4f}',
        'Improvement %': f'{improvement_pct:+.1f}%'
    })

df_comparison = pd.DataFrame(comparison_data).set_index('Metric')

print('\n' + '='*100)
print('FINAL RESULTS: Baseline vs EAGF (10-Seed Paired Evaluation)')
print('='*100)
print(df_comparison.to_string())
print('='*100)
print(f'\nResults based on {len(paired_seeds)} paired seeds: {paired_seeds}')


## 4. Statistical Analysis & Key Findings

In [ ]:
from scipy import stats

# Compute paired t-test and Wilcoxon signed-rank test for TI
ti_baseline = baseline_stats['trust_index']['values']
ti_eagf = eagf_stats['trust_index']['values']

# Paired t-test
t_stat, t_pval = stats.ttest_rel(ti_eagf, ti_baseline)

# Wilcoxon signed-rank test
w_stat, w_pval = stats.wilcoxon(ti_eagf, ti_baseline, method='approx')

# Effect size (r = Z / sqrt(N))
z = stats.norm.ppf(1 - w_pval/2)
effect_size = z / np.sqrt(len(paired_seeds))

# Confidence intervals (95%)
from scipy import stats as sp_stats
ci_baseline = sp_stats.t.interval(0.95, len(ti_baseline)-1, 
                                   loc=np.mean(ti_baseline), 
                                   scale=sp_stats.sem(ti_baseline))
ci_eagf = sp_stats.t.interval(0.95, len(ti_eagf)-1, 
                              loc=np.mean(ti_eagf), 
                              scale=sp_stats.sem(ti_eagf))

print('Statistical Analysis - Trust Index (TI)')
print('=' * 80)
print(f'\n  Baseline TI:  {np.mean(ti_baseline):.6f} ± {np.std(ti_baseline):.6f}')
print(f'               95% CI: [{ci_baseline[0]:.6f}, {ci_baseline[1]:.6f}]')
print(f'\n  EAGF TI:      {np.mean(ti_eagf):.6f} ± {np.std(ti_eagf):.6f}')
print(f'               95% CI: [{ci_eagf[0]:.6f}, {ci_eagf[1]:.6f}]')
print(f'\n  Improvement:  +{(np.mean(ti_eagf) - np.mean(ti_baseline)):.6f} ({(np.mean(ti_eagf)/np.mean(ti_baseline)-1)*100:.2f}%)')
print(f'\n  Paired t-test (TI):       t = {t_stat:.4f}, p = {t_pval:.6f}')
print(f'  Wilcoxon signed-rank:     W = {w_stat:.4f}, p = {w_pval:.6f}')
print(f'  Effect size (r):          r = {effect_size:.6f} (large effect)')
print(f'  Number of paired seeds:   n = {len(paired_seeds)}')
print('=' * 80)

# Key findings
print('\nKey Findings:')
print('-' * 80)
rp_improve = eagf_stats['recall_parity']['mean'] - baseline_stats['recall_parity']['mean']
print(f"  1. Fairness improvement (Recall Parity):")
print(f"     Baseline RP: {baseline_stats['recall_parity']['mean']:.4f}")
print(f"     EAGF RP:     {eagf_stats['recall_parity']['mean']:.4f}")
print(f"     Improvement: +{rp_improve:.4f} ✓")

print(f"\n  2. Trust Index improvement (statistically significant at α=0.05):")
print(f"     Wilcoxon p-value: {w_pval:.6f} {'✓ SIGNIFICANT' if w_pval < 0.05 else '✗ NOT SIGNIFICANT'}")
print(f"     Relative improvement: +{(np.mean(ti_eagf)/np.mean(ti_baseline)-1)*100:.2f}%")

print(f"\n  3. TI_certified (governance constraint):")
print(f"     Baseline: {np.mean([baseline_results[s].get('trust_index_certified', 0) for s in paired_seeds]):.4f}")
print(f"     EAGF:     {np.mean([eagf_results[s].get('trust_index_certified', 0) for s in paired_seeds]):.4f}")
print(f"     Note: TI_certified = 0 if any pillar below threshold (governance gating)")


## 5. Comparison Visualization: Baseline vs EAGF

In [ ]:
import matplotlib.pyplot as plt
import os

# Prepare data for comparison chart
metric_plot = ['Accuracy', 'Recall\nParity', 'Clarity\n(C)', 'Privacy\n(P)', 
               'Accountability\n(A)', 'Trust Index\n(TI)']
metric_keys = ['accuracy', 'recall_parity', 'clarity', 'privacy', 'accountability', 'trust_index']

baseline_vals = [baseline_stats[k]['mean'] for k in metric_keys]
eagf_vals = [eagf_stats[k]['mean'] for k in metric_keys]
baseline_errs = [baseline_stats[k]['std'] for k in metric_keys]
eagf_errs = [eagf_stats[k]['std'] for k in metric_keys]

# Create figure
x = np.arange(len(metric_keys))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars_baseline = ax.bar(x - w/2, baseline_vals, w, yerr=baseline_errs, 
                        label='Baseline (AIF360-DP)', color='#FF6B6B', 
                        edgecolor='white', linewidth=1, capsize=5, alpha=0.85)
bars_eagf = ax.bar(x + w/2, eagf_vals, w, yerr=eagf_errs,
                    label='EAGF', color='#4ECDC4', 
                    edgecolor='white', linewidth=1, capsize=5, alpha=0.85)

# Add value labels on bars
for bar, val in zip(bars_baseline, baseline_vals):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar, val in zip(bars_eagf, eagf_vals):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Metrics', fontsize=11, fontweight='bold')
ax.set_ylabel('Score', fontsize=11, fontweight='bold')
ax.set_title('EAGF Final Results: Baseline vs Framework (10-Seed Paired Evaluation)', 
             fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_plot, fontsize=10)
ax.set_ylim(0, 1.15)
ax.legend(loc='upper left', fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
os.makedirs(os.path.join(PROJECT_ROOT, 'figures'), exist_ok=True)
fig_path = os.path.join(PROJECT_ROOT, 'figures', 'notebook1_final_results.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')


## 6. Trust Index Component Breakdown (Baseline vs EAGF)

In [ ]:
# Create comparison chart of TI components (Clarity, Recall Parity, Privacy, Accountability)
pillars = ['Clarity\n(C)', 'Recall Parity\n(RP)', 'Privacy\n(P)', 'Accountability\n(A)']
pillar_keys = ['clarity', 'recall_parity', 'privacy', 'accountability']

baseline_pillars = [baseline_stats[k]['mean'] for k in pillar_keys]
eagf_pillars = [eagf_stats[k]['mean'] for k in pillar_keys]
baseline_pillar_errs = [baseline_stats[k]['std'] for k in pillar_keys]
eagf_pillar_errs = [eagf_stats[k]['std'] for k in pillar_keys]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline components
ax = axes[0]
bars_baseline = ax.barh(pillars, baseline_pillars, xerr=baseline_pillar_errs, 
                         color='#FF6B6B', edgecolor='white', capsize=5, alpha=0.85)
for bar, val in zip(bars_baseline, baseline_pillars):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 1.15)
ax.set_xlabel('Score', fontsize=11, fontweight='bold')
ax.set_title(f'Baseline TI Components\nTI = {baseline_stats["trust_index"]["mean"]:.4f}', 
             fontsize=11, fontweight='bold')
ax.axvline(1.0, color='grey', linestyle='--', alpha=0.4, linewidth=1.5)
ax.grid(axis='x', alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)

# EAGF components
ax = axes[1]
bars_eagf = ax.barh(pillars, eagf_pillars, xerr=eagf_pillar_errs,
                     color='#4ECDC4', edgecolor='white', capsize=5, alpha=0.85)
for bar, val in zip(bars_eagf, eagf_pillars):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 1.15)
ax.set_xlabel('Score', fontsize=11, fontweight='bold')
ax.set_title(f'EAGF TI Components\nTI = {eagf_stats["trust_index"]["mean"]:.4f}', 
             fontsize=11, fontweight='bold')
ax.axvline(1.0, color='grey', linestyle='--', alpha=0.4, linewidth=1.5)
ax.grid(axis='x', alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Trust Index Component Comparison (10-Seed Means with ±1 SD)', 
             fontsize=12, fontweight='bold', y=1.00)
plt.tight_layout()
fig_path2 = os.path.join(PROJECT_ROOT, 'figures', 'notebook1_ti_components.png')
plt.savefig(fig_path2, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path2}')


---
## Summary

### Final Verified Results (10-Seed Paired Evaluation)

| Finding | Value | Status |
|---|---|---|
| Baseline Trust Index (TI) | 0.6879 ± 0.0210 | ✓ |
| EAGF Trust Index (TI) | 0.8554 ± 0.0101 | ✓ |
| TI Improvement | +24.35% | ✓ |
| Statistical Significance (Wilcoxon p-value) | 0.005062 | ✓ p < 0.05 |
| Effect Size | r = 0.886 | ✓ Large |
| Paired Seeds | 10 (42-51) | ✓ |

### Key Metrics

**Fairness (Recall Parity):** Baseline 0.9333 → EAGF 1.0000 (+6.67%)  
**Clarity:** Baseline 0.6447 → EAGF 0.7231 (+12.18%)  
**Privacy:** Baseline 0.7292 → EAGF 0.8471 (+16.17%)  
**Accountability:** Baseline 0.3333 → EAGF 1.0000 (+200%)  

**TI_certified (Governance Constraint):** 0.0000 for both (no model meets all per-pillar thresholds)

### Next Notebooks

- `02_statistical_analysis.ipynb` — Statistical tests with significance and confidence intervals
- `03_reiot_fairness.ipynb` — RE-IoT domain-specific fairness analysis
- `04_pareto_front.ipynb` — Multi-objective Pareto front visualization
- `05_trust_index_sensitivity.ipynb` — TI weight sensitivity analysis and TI_certified exploration